# causal/02 DID 双重差分 实战

数据：Card & Krueger (1994) 公开均值汇总
目标：用 2x2 DID 估计 NJ 最低工资上涨对快餐就业的影响

三步法：
1. 算出 NJ/PA 政策前/后四个均值
2. 算 NJ 前后差、PA 前后差
3. 两个差再相减得到 DID

In [1]:
import pandas as pd

# Card & Krueger (1994) AER 84(4) Table 2 'All Stores' column.
# 4-row mean FTE employment: NJ/PA x pre/post NJ minimum wage hike ($4.25 -> $5.05).
# Microdata (410 stores x 2 waves) is at https://davidcard.berkeley.edu/data_sets.html
# (file njmin.zip -> public.dat). We use the published aggregated means here.
_card_krueger_rows = [
    # state, period, mean_fte, n_stores
    ('NJ', 'pre',  20.44, 331),
    ('NJ', 'post', 21.03, 331),
    ('PA', 'pre',  23.33,  79),
    ('PA', 'post', 21.17,  79),
]
df = pd.DataFrame(_card_krueger_rows, columns=['state', 'period', 'mean_fte', 'n_stores'])

def compute_did_2x2(df):
    """2x2 difference-in-differences on (NJ/PA) x (pre/post).

    Falls back to 'empfte' if 'mean_fte' is absent (for microdata use).
    """
    col = 'mean_fte' if 'mean_fte' in df.columns else 'empfte'
    nj_pre  = df[(df['state'] == 'NJ') & (df['period'] == 'pre')][col].mean()
    nj_post = df[(df['state'] == 'NJ') & (df['period'] == 'post')][col].mean()
    pa_pre  = df[(df['state'] == 'PA') & (df['period'] == 'pre')][col].mean()
    pa_post = df[(df['state'] == 'PA') & (df['period'] == 'post')][col].mean()
    nj_diff = nj_post - nj_pre
    pa_diff = pa_post - pa_pre
    did = nj_diff - pa_diff
    return {
        'nj_pre': nj_pre, 'nj_post': nj_post,
        'pa_pre': pa_pre, 'pa_post': pa_post,
        'nj_diff': nj_diff, 'pa_diff': pa_diff,
        'did': did,
    }

print('Card & Krueger 1994 mean FTE:')
print(df.to_string(index=False))
n_nj = df[df['state']=='NJ']['n_stores'].iloc[0]
n_pa = df[df['state']=='PA']['n_stores'].iloc[0]
print(f'\nUnique stores: NJ {n_nj} + PA {n_pa} = {n_nj + n_pa} (each surveyed in two waves)')

Card & Krueger 1994 mean FTE:
state period  mean_fte  n_stores
   NJ    pre     20.44       331
   NJ   post     21.03       331
   PA    pre     23.33        79
   PA   post     21.17        79

Unique stores: NJ 331 + PA 79 = 410 (each surveyed in two waves)


## 数据概览

NJ 在 1992 年 4 月 1 日将最低工资从 $4.25 提到 $5.05。PA 全年保持 $4.25。两州的快餐行业（Burger King、KFC、Wendy's、Roy Rogers）相似度高，适合做对照。

Card & Krueger 选了 NJ 331 家、PA 79 家快餐店，在 1992 年 2 月（政策前）和 1992 年 11 月（政策后）各做了一次电话调查。这是 DID 方法在劳动经济学的开山实证。

In [2]:
## Step 1: 2x2 均值表

# Pivot to 2x2 view
pivot = df.pivot(index='state', columns='period', values='mean_fte')
print('2x2 table (mean FTE employment):')
print(pivot.round(2))
print()
print('NJ pre  / NJ post:', pivot.loc['NJ', 'pre'], '/', pivot.loc['NJ', 'post'])
print('PA pre  / PA post:', pivot.loc['PA', 'pre'], '/', pivot.loc['PA', 'post'])

2x2 table (mean FTE employment):
period   post    pre
state               
NJ      21.03  20.44
PA      21.17  23.33

NJ pre  / NJ post: 20.44 / 21.03
PA pre  / PA post: 23.33 / 21.17


## Step 2: 前后差（per-group difference）

NJ 自己的前后差 = NJ 后 - NJ 前，反映 NJ 时期的所有变化（包括最低工资上涨 + 季节性 + 宏观经济）。

PA 自己的前后差 = PA 后 - PA 前，反映 PA 时期的所有变化（季节性 + 宏观经济，但没有最低工资上涨）。

In [3]:
## Step 2: 前/后差
nj_diff = pivot.loc['NJ', 'post'] - pivot.loc['NJ', 'pre']
pa_diff = pivot.loc['PA', 'post'] - pivot.loc['PA', 'pre']
print(f'NJ diff (post - pre): {nj_diff:+.2f}')
print(f'PA diff (post - pre): {pa_diff:+.2f}')

NJ diff (post - pre): +0.59
PA diff (post - pre): -2.16


## Step 3: 差之差（diff-in-diff）

DID = NJ 差 - PA 差。

直觉：NJ 的前后变化里包含了"季节性 + 宏观经济"，PA 的前后变化里也包含了"季节性 + 宏观经济"（假设）。两个差再减一次，把这些共同的随时间变化剥离掉，剩下的就是最低工资上涨的真实影响。

这个差就是**对处理组的平均因果效应 ATT**。

In [4]:
## Step 3: 差之差
did = nj_diff - pa_diff
print(f'DID estimate (NJ diff - PA diff): {did:+.2f}')
print()
print('Interpretation: NJ 提最低工资后，快餐就业 FTE 比 PA 反向')
print('增加了 +2.75 个职位（per restaurant）。与经典经济学预测')
print('"涨最低工资 -> 裁员" 相反。这是 DID 在劳动经济学最经典的')
print('实证发现。')

print()
print('Verify with helper function:')
res = compute_did_2x2(df)
for k, v in res.items():
    print(f'  {k}: {v:+.2f}')

DID estimate (NJ diff - PA diff): +2.75

Interpretation: NJ 提最低工资后，快餐就业 FTE 比 PA 反向
增加了 +2.75 个职位（per restaurant）。与经典经济学预测
"涨最低工资 -> 裁员" 相反。这是 DID 在劳动经济学最经典的
实证发现。

Verify with helper function:
  nj_pre: +20.44
  nj_post: +21.03
  pa_pre: +23.33
  pa_post: +21.17
  nj_diff: +0.59
  pa_diff: -2.16
  did: +2.75


## 平行趋势假设

DID 的核心假设：若 NJ 不涨最低工资，NJ 与 PA 的就业会同步变化。

**怎么直觉检验？** 看政策前 NJ 与 PA 是否有共同趋势。Card & Krueger 的调查只有两波，但其他研究的多次调查（Neumark & Wascher 2000 等）显示 NJ 与 PA 政策前趋势确实接近平行。

**怎么补救？** 加协变量、做事件研究、加个体固定效应。

## 实战小结

- NJ 政策后 FTE 增加了 0.59，PA 减少了 2.16
- 差之差 **+2.75** = NJ 提最低工资对就业的因果效应
- 这与经典理论预测相反，催生了大量后续"最低工资是否真的裁员"的研究
- 局限：只有两波调查，无法严格画政策前趋势；且样本只覆盖东 PA 快餐店，外推到全国需谨慎

**注意**：本文用的是论文公开的均值汇总。原始 410 家店面板在 Card 主页公开，作者推荐从 David Card 数据集页面下载 `public.dat` 做更细致的分析。